# N01 — Tokenização: como o texto vira números

**Capítulo-âncora:** C3 — Tokens · **Invariante:** Inv. 5 (Custo Composto)

## O que este notebook resolve

Você já ouviu mil vezes que LLMs operam em tokens, não em palavras. Mas até ver na própria máquina como **'Internacionalização' vira 4 tokens** e **'cat' vira 1**, custo composto continua abstração. Este notebook materializa isso em quatro experimentos curtos, todos rodando localmente, sem chamada de API, sem custo.

Ao fim, você sai sabendo:
- Por que português custa 30-50% mais que inglês na mesma tradução
- Como contar tokens **antes** de enviar (e quanto vai custar)
- Por que código costuma tokenizar pior que prosa
- Como BPE (Byte Pair Encoding) escolhe onde quebrar


## Setup — só uma biblioteca, sem chave de API


In [ ]:
# Tiktoken é o tokenizador open-source que reproduz o BPE do GPT.
# Usado aqui como proxy didático; Claude usa um tokenizador próprio similar.
import tiktoken

# Carrega o codificador padrão moderno
enc = tiktoken.get_encoding('cl100k_base')
print(f'Encoding carregado: {enc.name}')
print(f'Vocabulário: {enc.n_vocab:,} tokens')


## Experimento 1 — Ver os tokens com os próprios olhos

Pegue uma frase em português. O método `encode()` devolve os IDs dos tokens; o método `decode_single_token_bytes()` mostra o pedaço de texto que cada ID representa.


In [ ]:
texto = 'A inteligência artificial transformou a economia digital.'
tokens = enc.encode(texto)

print(f'Texto: {texto!r}')
print(f'Total de tokens: {len(tokens)}')
print(f'Total de caracteres: {len(texto)}')
print(f'Razão caracteres/token: {len(texto)/len(tokens):.2f}')
print()
print('Decomposição token a token:')
for tid in tokens:
    piece = enc.decode_single_token_bytes(tid).decode('utf-8', errors='replace')
    print(f'  {tid:>6}  →  {piece!r}')


**Observe:** palavras curtas e comuns ('A', 'a') viram um token. Palavras longas como 'inteligência' e 'transformou' se quebram em vários pedaços. A divisão não respeita morfologia — segue frequência estatística no corpus de treino.


## Experimento 2 — Português custa mais que inglês


In [ ]:
pares = [
    ('Olá, como vai você hoje?', 'Hello, how are you today?'),
    ('Inteligência artificial', 'Artificial intelligence'),
    ('Desenvolvimento de software corporativo', 'Enterprise software development'),
    ('Implementação institucionalizada', 'Institutionalized implementation'),
]

print(f'{"Português":<45} {"PT tok":>7} {"EN tok":>7} {"Δ%":>7}')
print('-' * 75)
for pt, en in pares:
    pt_t = len(enc.encode(pt))
    en_t = len(enc.encode(en))
    delta = (pt_t - en_t) / en_t * 100
    print(f'{pt:<45} {pt_t:>7} {en_t:>7} {delta:>+6.0f}%')


**Conclusão prática:** o mesmo conteúdo em português custa, em média, 30-50% mais tokens que em inglês. Em operação de escala, isso é a diferença entre orçamento honesto e estouro de fim de mês. Aplicação direta do Princípio 5 (Custo Composto): o multiplicador 'língua' entra na fórmula F7.


## Experimento 3 — Código tokeniza pior que prosa


In [ ]:
prosa = 'Esta é uma frase normal em português com palavras comuns do cotidiano.'
codigo = '''def calcular_juros_compostos(principal, taxa, tempo):
    return principal * ((1 + taxa) ** tempo - 1)'''

print(f'Prosa  ({len(prosa)} chars): {len(enc.encode(prosa))} tokens — {len(prosa)/len(enc.encode(prosa)):.2f} c/t')
print(f'Código ({len(codigo)} chars): {len(enc.encode(codigo))} tokens — {len(codigo)/len(enc.encode(codigo)):.2f} c/t')


Código tem identificadores únicos (`calcular_juros_compostos`), pontuação densa (parênteses, colchetes, dois-pontos) e indentação significativa. Tudo isso custa tokens. Por isso prompts com código exigem orçamento próprio — não use a heurística 'um token ≈ uma palavra' para repositórios.


## Experimento 4 — Estimar custo antes de enviar


In [ ]:
# Tarifário ILUSTRATIVO (consulte o Apêndice J do livro para versão corrente).
# Sonnet ~ input $3/Mtok, output $15/Mtok em janeiro/2026.
PRICE_IN_PER_MTOK = 3.00
PRICE_OUT_PER_MTOK = 15.00

def estimar(prompt: str, resposta_estimada_tokens: int = 500) -> dict:
    in_tokens = len(enc.encode(prompt))
    in_cost = (in_tokens / 1_000_000) * PRICE_IN_PER_MTOK
    out_cost = (resposta_estimada_tokens / 1_000_000) * PRICE_OUT_PER_MTOK
    return {
        'tokens_in': in_tokens,
        'tokens_out_esperado': resposta_estimada_tokens,
        'custo_in_usd': round(in_cost, 6),
        'custo_out_usd': round(out_cost, 6),
        'custo_total_usd': round(in_cost + out_cost, 6),
        'custo_brl_aprox': round((in_cost + out_cost) * 5.0, 4),
    }

prompt_exemplo = '''Você é um analista jurídico. Leia o contrato abaixo e identifique cláusulas de risco.
Para cada cláusula identificada, devolva: trecho exato, tipo de risco, recomendação.

CONTRATO:
[... contrato simulado de 2000 palavras seria colado aqui ...]'''

resultado = estimar(prompt_exemplo, resposta_estimada_tokens=800)
for k, v in resultado.items():
    print(f'{k:>22}: {v}')


**Custo por chamada é trivial. Multiplicado pelo número de chamadas, vira a conta real.** Se este prompt rodar 10.000 vezes por mês, multiplique. Esse é o Princípio 5 em código: o preço do token é só o primeiro andar da fórmula.


## O que fazer com isso

1. **Toda vez que você for desenhar um prompt ou agente, conte os tokens antes de operar em escala.**
2. **Em prompts em português, considere se vale a pena escrever a parte fixa em inglês** (constituição, instruções) e a parte variável no idioma do usuário. Custo composto cai 20-30% sem perda de qualidade.
3. **Antes de qualquer contrato com fornecedor de IA, faça o cálculo desta célula com a sua carga real.** Tarifário publicado é por token; sua conta é por (token × chamada × usuário × redundância × tier).

**Próximo notebook:** [N02 — Janela de Contexto e Lost in the Middle](./n02-janela-contexto.ipynb), onde a posição do token dentro do prompt vira variável crítica.
